# 03 - LIME and SHAP XAI Extension

Purpose: add optional LIME/SHAP comparison scaffolding beside the existing CAM-family notebooks. This is a research extension only; Grad-CAM remains the primary production explanation method.

Install optional dependencies only if you plan to run this notebook:

```bash
pip install lime shap
```

In [4]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
RESEARCH_DIR = NOTEBOOK_DIR.parent
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

from research_paths import METADATA_PATH, MODEL_DIR

METRICS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"
FIGURES_DIR = NOTEBOOK_DIR / "outputs" / "figures"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
try:
    import lime
    from lime import lime_image
    LIME_AVAILABLE = True
except Exception as exc:
    LIME_AVAILABLE = False
    print("LIME unavailable:", exc)

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    SHAP_AVAILABLE = False
    print("SHAP unavailable:", exc)

print({"lime": LIME_AVAILABLE, "shap": SHAP_AVAILABLE})

{'lime': True, 'shap': True}


## Implementation Plan

Use this notebook after a trained ResNet50 checkpoint exists.

1. Sample 10-25 correctly classified HAM10000 images.
2. Generate Grad-CAM using existing RQ1 helpers or `pytorch-grad-cam`.
3. Generate LIME superpixel explanations.
4. Generate SHAP image attributions on a very small sample because runtime is high.
5. Compare each method with the same summary metrics where possible:
   - focus area percentage after thresholding
   - entropy / concentration
   - qualitative agreement with lesion region
   - runtime per image

Do not use LIME/SHAP output directly in production until runtime, stability, and medical-language guardrails are validated.

In [6]:
import pandas as pd

rows = [
    {"method": "GradCAM", "status": "primary", "production_candidate": True, "notes": "Already evaluated in RQ1/RQ2."},
    {"method": "LIME", "status": "optional_research", "production_candidate": False, "notes": "Useful for local superpixel perturbation comparison; slower and less stable."},
    {"method": "SHAP", "status": "optional_research", "production_candidate": False, "notes": "Useful for attribution comparison; expensive for images."},
]
summary = pd.DataFrame(rows)
summary.to_csv(METRICS_DIR / "XAI_lime_shap_extension_plan.csv", index=False)
summary

,method,status,production_candidate,notes
0,GradCAM,primary,True,Already evaluated in RQ1/RQ2.
1,LIME,optional_research,False,Useful for local superpixel perturbation compa...
2,SHAP,optional_research,False,Useful for attribution comparison; expensive f...
